In [ ]:
from pathlib import Path
import pandas as pd

# Update this to the correct absolute path if different
FILEPATH = Path(r"d:\\Celestial\\preparation\\edunet_project\\netflix\\Netflix_Dataset_1.csv")

# Check file exists before attempting to read
if not FILEPATH.is_file():
    raise FileNotFoundError(f"Dataset not found at: {FILEPATH}")

try:
    df = pd.read_csv(FILEPATH)
except Exception as e:
    raise RuntimeError(f"Failed to read Excel file: {e}")

print("Loaded OK. Columns:", df.columns.tolist())
print(df.head())

# Print info (remove or comment out on very large datasets)
print(df.info())

### Dataset load output

Below is the captured output from running the first cell (dataframe info and first 5 rows):

```
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7789 entries, 0 to 7788
Data columns (total 11 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   Show_Id       7789 non-null   object
 1   Category      7789 non-null   object
 2   Title         7789 non-null   object
 3   Director      5401 non-null   object
 4   Cast          7071 non-null   object
 5   Country       7282 non-null   object
 6   Release_Date  7779 non-null   object
 7   Rating        7782 non-null   object
 8   Duration      7789 non-null   object
 9   Type          7789 non-null   object
 10  Description   7789 non-null   object
dtypes: object(11)
memory usage: 669.5+ KB

=== HEAD (first 5 rows) ===
  Show_Id Category  Title           Director                                                                                                                                                                        Cast        Country       Release_Date Rating   Duration                                                      Type                                                                                                                                            Description
0      s1  TV Show     3%                NaN  Jo�o Miguel, Bianca Comparato, Michel Gomes, Rodolfo Valente, Vaneza Oliveira, Rafael Lozano, Viviane Porto, Mel Fronckowiak, Sergio Mamberti, Zez� Motta, Celso Frateschi         Brazil    August 14, 2020  TV-MA  4 Seasons    International TV Shows, TV Dramas, TV Sci-Fi & Fantasy               In a future where the elite inhabit an island paradise far from the crowded slums, you get one chance to join the 3% saved from squalor.
1      s2    Movie  07:19  Jorge Michel Grau                                                                                    Demi�n Bichir, H�ctor Bonilla, Oscar Serrano, Azalia Ortiz, Octavio Michel, Carmen Beato         Mexico  December 23, 2016  TV-MA     93 min                              Dramas, International Movies   After a devastating earthquake hits Mexico City, trapped survivors from all walks of life wait to be rescued while trying desperately to stay alive.
2      s3    Movie  23:59       Gilbert Chan                                                                Tedd Chan, Stella Chung, Henley Hii, Lawrence Koh, Tommy Kuan, Josh Lai, Mark Lee, Susan Leong, Benjamin Lim      Singapore  December 20, 2018      R     78 min                       Horror Movies, International Movies  When an army recruit is found dead, his fellow soldiers are forced to confront a terrifying secret that's haunting their jungle island training camp.
3      s4    Movie      9        Shane Acker                             Elijah Wood, John C. Reilly, Jennifer Connelly, Christopher Plummer, Crispin Glover, Martin Landau, Fred Tatasciore, Alan Oppenheimer, Tom Kane  United States  November 16, 2017  PG-13     80 min  Action & Adventure, Independent Movies, Sci-Fi & Fantasy      In a postapocalyptic world, rag-doll robots hide in fear from dangerous machines out to exterminate them, until a brave newcomer joins the group.
4      s5    Movie     21     Robert Luketic             Jim Sturgess, Kevin Spacey, Kate Bosworth, Aaron Yoo, Liza Lapira, Jacob Pitts, Laurence Fishburne, Jack McGee, Josh Gad, Sam Golzari, Helen Carey, Jack Gilpin  United States    January 1, 2020  PG-13    123 min                                                    Dramas        A brilliant group of students become card-counting experts with the intent of swindling millions out of Las Vegas casinos by playing blackjack.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Ensure required columns exist
required = ['Release_Date', 'Category']
missing = [c for c in required if c not in df.columns]
if missing:
    raise KeyError(f"Missing required columns in dataframe: {missing}")

# Parse dates safely and extract year
df_cleaned = df.copy()
df_cleaned['Release_Date'] = pd.to_datetime(df_cleaned['Release_Date'], errors='coerce')
df_cleaned['Release_Year'] = df_cleaned['Release_Date'].dt.year
df_cleaned = df_cleaned.dropna(subset=['Release_Year']).copy()

# Group by year and category and count the titles
content_by_year = df_cleaned.groupby(['Release_Year', 'Category']).size().unstack(fill_value=0)

# Sort the index (years) and filter from 2008 onwards for a clearer trend
content_by_year = content_by_year.sort_index()
content_by_year = content_by_year[content_by_year.index >= 2008]

if content_by_year.empty:
    print('No data available after filtering (2008+). Skipping plot.')
else:
    ax = content_by_year.plot(kind='bar', stacked=True, figsize=(12, 6))
    ax.set_title('Distribution of Movies vs. TV Shows Added Over the Years (2008-2021)')
    ax.set_xlabel('Release Year')
    ax.set_ylabel('Count of Titles Added')
    plt.xticks(rotation=45, ha='right')
    ax.legend(title='Category')
    plt.tight_layout()
    plt.savefig('content_distribution_over_years.png')
    plt.close()

# Generated plot
Below is the plot generated by the analysis:

![Distribution of Movies vs TV Shows](content_distribution_over_years.png)

*Image file saved at `content_distribution_over_years.png` in the notebook directory.*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Create a copy to work with the cleaned data
df_genres = df_cleaned.copy()
# Split the 'Type' column into a list of genres
df_genres['Genre'] = df_genres['Type'].str.split(', ')

# Explode the DataFrame to have one row per genre
df_exploded = df_genres.explode('Genre')

# Count the frequency of each genre
genre_counts = df_exploded['Genre'].value_counts()

# Identify the top 10 most common genres
top_10_genres = genre_counts.head(10)

# Create a bar chart for the top 10 genres
plt.figure(figsize=(12, 6))
top_10_genres.sort_values(ascending=True).plot(kind='barh', color='skyblue')
plt.title('Top 10 Most Common Genres on Netflix')
plt.xlabel('Count of Titles')
plt.ylabel('Genre')
plt.tight_layout()
plt.savefig('top_10_genres.png')
plt.close()

# Top 10 Genres
Below is the top-10 genres bar chart generated by the analysis:

![Top 10 Genres](top_10_genres.png)

*Image file saved at `top_10_genres.png` in the notebook directory.*

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Drop rows with missing 'Country'
df_country = df.dropna(subset=['Country']).copy()

# Split the 'Country' column into individual countries
df_country['Country'] = df_country['Country'].str.split(', ')

# Explode the DataFrame to have one row per country
df_exploded_country = df_country.explode('Country')

# Count the frequency of each country
country_counts = df_exploded_country['Country'].value_counts()

# Identify the top 10 contributing countries
top_10_countries = country_counts.head(10)

# Create a bar chart for the top 10 countries
plt.figure(figsize=(12, 6))
top_10_countries.sort_values(ascending=True).plot(kind='barh', color='lightcoral')
plt.title('Top 10 Contributing Countries to Netflix Catalog')
plt.xlabel('Count of Titles')
plt.ylabel('Country')
plt.tight_layout()
plt.savefig('top_10_countries.png')
plt.close()

# Top 10 Contributing Countries
Below is the top-10 countries bar chart generated by the analysis:

![Top 10 Countries](top_10_countries.png)

*Image file saved at `top_10_countries.png` in the notebook directory.*